Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы. Отчёт будет отражать динамику изменения популярности языков программирования и представлять собой набор таблиц "топ-10" для каждого года.

Получившийся отчёт сохранить в формате Apache Parquet.

In [1]:
!pip3 install pyspark==3.1.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.4/212.4 MB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.6/198.6 kB 16.1 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.1.2-py2.py3-none-any.whl size=212880750 sha256=c71a2bf2343a353ce5f1e5f5701b79af055c251c6a859e7afebbb89315c8ef93
  Stored in directory: /root/.cache/pip/wheels/ab/65/b0/6a8019874e2b98d34b730b15d4a84bf226f17ef1679d9f451e
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conn

In [21]:
import os
import sys
import pyspark.sql.functions as F
from pyspark.sql import Row
from pyspark.sql import SparkSession
from pyspark.sql.window import Window

In [3]:
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.12:0.17.0 pyspark-shell'

In [7]:
spark = SparkSession.builder.getOrCreate()
spark

In [5]:
!wget https://git.ai.ssau.ru/tk/big_data/raw/branch/master/data/posts_sample.xml

--2026-03-30 07:52:01--  https://git.ai.ssau.ru/tk/big_data/raw/branch/master/data/posts_sample.xml
Resolving git.ai.ssau.ru (git.ai.ssau.ru)... 91.222.131.161
Connecting to git.ai.ssau.ru (git.ai.ssau.ru)|91.222.131.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 74162295 (71M) [text/plain]
Saving to: ‘posts_sample.xml’

posts_sample.xml    100%[===================>]  70.73M   235KB/s    in 4m 35s  

2026-03-30 07:56:37 (264 KB/s) - ‘posts_sample.xml’ saved [74162295/74162295]



In [8]:
postsData = spark.read.format('xml').option('rowTag', 'row').option("timestampFormat", 'y/M/d H:m:s').load('posts_sample.xml')

In [9]:
# Структура данных
print("--- СХЕМА ДАННЫХ ---")
postsData.printSchema()

# Первые 5 строк
print("--- ПЕРВЫЕ 5 ЭЛЕМЕНТОВ ---")
postsData.show(5, truncate=False)

# Статистическое описание
print("--- ОПИСАНИЕ ДАТАСЕТА ---")
postsData.describe().show()

# Общее количество записей
print(f"Количество строк: {postsData.count()}")

--- СХЕМА ДАННЫХ ---
root
 |-- _AcceptedAnswerId: long (nullable = true)
 |-- _AnswerCount: long (nullable = true)
 |-- _Body: string (nullable = true)
 |-- _ClosedDate: timestamp (nullable = true)
 |-- _CommentCount: long (nullable = true)
 |-- _CommunityOwnedDate: timestamp (nullable = true)
 |-- _CreationDate: timestamp (nullable = true)
 |-- _FavoriteCount: long (nullable = true)
 |-- _Id: long (nullable = true)
 |-- _LastActivityDate: timestamp (nullable = true)
 |-- _LastEditDate: timestamp (nullable = true)
 |-- _LastEditorDisplayName: string (nullable = true)
 |-- _LastEditorUserId: long (nullable = true)
 |-- _OwnerDisplayName: string (nullable = true)
 |-- _OwnerUserId: long (nullable = true)
 |-- _ParentId: long (nullable = true)
 |-- _PostTypeId: long (nullable = true)
 |-- _Score: long (nullable = true)
 |-- _Tags: string (nullable = true)
 |-- _Title: string (nullable = true)
 |-- _ViewCount: long (nullable = true)

--- ПЕРВЫЕ 5 ЭЛЕМЕНТОВ ---
+-----------------+----------

In [10]:
# Ограничиваем выборку по временным рамкам: с начала 2010 по конец 2020
start_period = "2010-01-01"
end_period = "2020-12-31"

# Фильтрация данных по дате создания поста
filtered_posts = postsData.where(
    (F.col("_CreationDate") >= start_period) &
    (F.col("_CreationDate") <= end_period)
)

filtered_posts.show(10)

+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+-----+------+----------+
|_AcceptedAnswerId|_AnswerCount|               _Body|_ClosedDate|_CommentCount| _CommunityOwnedDate|       _CreationDate|_FavoriteCount|    _Id|   _LastActivityDate|       _LastEditDate|_LastEditorDisplayName|_LastEditorUserId|_OwnerDisplayName|_OwnerUserId|_ParentId|_PostTypeId|_Score|_Tags|_Title|_ViewCount|
+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+-----+------+----------+
|             null|        null|<p>No. (And more ...|       null

In [11]:
!wget https://git.ai.ssau.ru/tk/big_data/raw/branch/master/data/programming-languages.csv

--2026-03-30 07:58:31--  https://git.ai.ssau.ru/tk/big_data/raw/branch/master/data/programming-languages.csv
Resolving git.ai.ssau.ru (git.ai.ssau.ru)... 91.222.131.161
Connecting to git.ai.ssau.ru (git.ai.ssau.ru)|91.222.131.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 40269 (39K) [text/plain]
Saving to: ‘programming-languages.csv’

programming-languag 100%[===================>]  39.33K   140KB/s    in 0.3s    

2026-03-30 07:58:32 (140 KB/s) - ‘programming-languages.csv’ saved [40269/40269]



In [12]:
languagesData = spark.read.format('csv').option('header', 'true').option("inferSchema", True).load('programming-languages.csv').dropna()

In [13]:
# Структурный обзор данных
print("=== СХЕМА ДАТАСЕТА ЯЗЫКОВ ===")
languagesData.printSchema()

# Демонстрация выборки
print("=== ПЕРВЫЕ 5 ЗАПИСЕЙ ===")
languagesData.show(5, truncate=False)

# Статистическое резюме
print("=== СВОДКА ДАННЫХ ===")
languagesData.describe().show()

# Итоговое количество записей
print(f"Общее число языков в справочнике: {languagesData.count()}")

=== СХЕМА ДАТАСЕТА ЯЗЫКОВ ===
root
 |-- name: string (nullable = true)
 |-- wikipedia_url: string (nullable = true)

=== ПЕРВЫЕ 5 ЗАПИСЕЙ ===
+----------+---------------------------------------------------------+
|name      |wikipedia_url                                            |
+----------+---------------------------------------------------------+
|A# .NET   |https://en.wikipedia.org/wiki/A_Sharp_(.NET)             |
|A# (Axiom)|https://en.wikipedia.org/wiki/A_Sharp_(Axiom)            |
|A-0 System|https://en.wikipedia.org/wiki/A-0_System                 |
|A+        |https://en.wikipedia.org/wiki/A%2B_(programming_language)|
|A++       |https://en.wikipedia.org/wiki/A%2B%2B                    |
+----------+---------------------------------------------------------+
only showing top 5 rows

=== СВОДКА ДАННЫХ ===
+-------+--------+--------------------+
|summary|    name|       wikipedia_url|
+-------+--------+--------------------+
|  count|     699|                 699|
|   mean|   

In [14]:
# Получаем названия всех языков программирования из справочника
all_languages = list(map(lambda row: str(row['name']), languagesData.collect()))
all_languages

['A# .NET',
 'A# (Axiom)',
 'A-0 System',
 'A+',
 'A++',
 'ABAP',
 'ABC',
 'ABC ALGOL',
 'ABSET',
 'ABSYS',
 'ACC',
 'Accent',
 'Ace DASL',
 'ACL2',
 'ACT-III',
 'Action!',
 'ActionScript',
 'Ada',
 'Adenine',
 'Agda',
 'Agilent VEE',
 'Agora',
 'AIMMS',
 'Alef',
 'ALF',
 'ALGOL 58',
 'ALGOL 60',
 'ALGOL 68',
 'ALGOL W',
 'Alice',
 'Alma-0',
 'AmbientTalk',
 'Amiga E',
 'AMOS',
 'AMPL',
 'Apex (Salesforce.com)',
 'APL',
 "App Inventor for Android's visual block language",
 'AppleScript',
 'Arc',
 'ARexx',
 'Argus',
 'AspectJ',
 'Assembly language',
 'ATS',
 'Ateji PX',
 'AutoHotkey',
 'Autocoder',
 'AutoIt',
 'AutoLISP / Visual LISP',
 'Averest',
 'AWK',
 'Axum',
 'B',
 'Babbage',
 'Bash',
 'BASIC',
 'bc',
 'BCPL',
 'BeanShell',
 'Batch (Windows/Dos)',
 'Bertrand',
 'BETA',
 'Bigwig',
 'Bistro',
 'BitC',
 'BLISS',
 'Blockly',
 'BlooP',
 'Blue',
 'Boo',
 'Boomerang',
 'Bourne shell (including',
 'bash and',
 'ksh )',
 'BREW',
 'BPEL',
 'C',
 'C--',
 'C++ – ISO/IEC 14882',
 'C# – ISO/IEC

In [18]:
def identify_language(row):
    # Приводим теги к нижнему регистру один раз
    raw_tags = str(row._Tags).lower()

    # Ищем первое вхождение языка, обернутого в скобки <>
    match = next((lang for lang in all_languages
                  if f"<{lang.lower()}>" in raw_tags), "No")

    return (row[6], match)

In [22]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Используем широковещательную переменную (broadcast), чтобы не гонять данные по сети
lang_list = [row['name'].strip().lower() for row in languagesData.collect()]

# Очищаем теги от < > и превращаем их в массив
processed_df = filtered_posts.withColumn(
    "tags_array",
    F.split(F.regexp_replace(F.col("_Tags"), r'[<>]', ' '), ' ')
)

# Разворачиваем массив тегов в строки и очищаем от пустых значений
exploded_df = processed_df.withColumn("tag", F.explode(F.col("tags_array"))) \
    .filter(F.col("tag") != "") \
    .withColumn("tag", F.lower(F.trim(F.col("tag"))))

# Фильтруем только те теги, которые есть в нашем списке языков
final_tags_df = exploded_df.filter(F.col("tag").isin(lang_list))

# Извлекаем год из даты создания
result_df = final_tags_df.withColumn("Year", F.year("_CreationDate")) \
    .groupBy("Year", F.col("tag").alias("Language")) \
    .count()

# выбор ТОП-10 для каждого года с помощью оконной функции
window_spec = Window.partitionBy("Year").orderBy(F.desc("count"))

top_10_df = result_df.withColumn("Rank", F.row_number().over(window_spec)) \
    .filter(F.col("Rank") <= 10) \
    .select("Year", "Language", F.col("count").alias("Count")) \
    .orderBy(F.desc("Year"), F.desc("Count"))

top_10_df.show(50, truncate=False)

# сохранение в Parquet
top_10_df.write.mode("overwrite").parquet("top_10_languages_2010_2020.parquet")

+----+-----------+-----+
|Year|Language   |Count|
+----+-----------+-----+
|2019|python     |166  |
|2019|javascript |135  |
|2019|java       |95   |
|2019|php        |65   |
|2019|r          |37   |
|2019|typescript |17   |
|2019|c          |14   |
|2019|bash       |11   |
|2019|go         |9    |
|2019|matlab     |9    |
|2018|python     |220  |
|2018|javascript |198  |
|2018|java       |146  |
|2018|php        |111  |
|2018|r          |66   |
|2018|typescript |27   |
|2018|c          |24   |
|2018|scala      |23   |
|2018|powershell |13   |
|2018|bash       |12   |
|2017|javascript |246  |
|2017|java       |204  |
|2017|python     |193  |
|2017|php        |138  |
|2017|r          |56   |
|2017|c          |25   |
|2017|typescript |20   |
|2017|objective-c|19   |
|2017|ruby       |17   |
|2017|powershell |14   |
|2016|javascript |278  |
|2016|java       |184  |
|2016|php        |155  |
|2016|python     |146  |
|2016|r          |52   |
|2016|c          |32   |
|2016|ruby       |24   |


Проверка

In [24]:
# читаем сохраненный отчет
check_df = spark.read.parquet("top_10_languages_2010_2020.parquet")

# сортировка сначала по году (убывание), потом по количеству (убывание)
# так мы увидим самые популярные языки вверху для каждого года
check_df.orderBy(F.desc("Year"), F.desc("Count")).show(110, truncate=False)

+----+-----------+-----+
|Year|Language   |Count|
+----+-----------+-----+
|2019|python     |166  |
|2019|javascript |135  |
|2019|java       |95   |
|2019|php        |65   |
|2019|r          |37   |
|2019|typescript |17   |
|2019|c          |14   |
|2019|bash       |11   |
|2019|matlab     |9    |
|2019|go         |9    |
|2018|python     |220  |
|2018|javascript |198  |
|2018|java       |146  |
|2018|php        |111  |
|2018|r          |66   |
|2018|typescript |27   |
|2018|c          |24   |
|2018|scala      |23   |
|2018|powershell |13   |
|2018|bash       |12   |
|2017|javascript |246  |
|2017|java       |204  |
|2017|python     |193  |
|2017|php        |138  |
|2017|r          |56   |
|2017|c          |25   |
|2017|typescript |20   |
|2017|objective-c|19   |
|2017|ruby       |17   |
|2017|powershell |14   |
|2016|javascript |278  |
|2016|java       |184  |
|2016|php        |155  |
|2016|python     |146  |
|2016|r          |52   |
|2016|c          |32   |
|2016|ruby       |24   |
